In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

chave = ["SG_UF", "CD_MUNICIPIO", "NR_ZONA", "NR_SECAO"]
codigo_sao_paulo = "71072"

In [17]:
locais_path = '../dados/locais/eleitorado_local_votacao_2026_SP.csv'

b_colunas = chave + [
    "NR_TURNO",
    "NM_MUNICIPIO",
    "NM_BAIRRO",
    "DS_TIPO_SECAO_AGREGADA",
    "NR_LOCAL_VOTACAO",
    "QT_ELEITOR_SECAO",
]

bairros = pd.read_csv(locais_path, sep=';', encoding='latin-1', usecols=b_colunas, dtype={"CD_MUNICIPIO": str})
bairros = bairros.loc[(bairros['CD_MUNICIPIO'].eq(codigo_sao_paulo)) & 
                      (bairros['NR_TURNO'].eq(1) &
                       bairros['DS_TIPO_SECAO_AGREGADA'].eq("Principal"))].copy()

assert not bairros.duplicated(chave).any()
print(f"Seções Principais do município de São Paulo: {len(bairros)}.")
print(f"Número de bairros distintos: {bairros['NM_BAIRRO'].nunique()}.")
bairros.head(10)

Seções Principais do município de São Paulo: 26683.
Número de bairros distintos: 899.


,NR_TURNO,SG_UF,CD_MUNICIPIO,NM_MUNICIPIO,NR_ZONA,NR_SECAO,DS_TIPO_SECAO_AGREGADA,NR_LOCAL_VOTACAO,NM_BAIRRO,QT_ELEITOR_SECAO
5,1,SP,71072,SÃO PAULO,422,108,Principal,1074,VILA AMALIA (ZONA NORTE),376
6,1,SP,71072,SÃO PAULO,349,223,Principal,1171,VILA GUSTAVO,342
7,1,SP,71072,SÃO PAULO,2,142,Principal,1090,HIGIENOPOLIS,373
10,1,SP,71072,SÃO PAULO,326,867,Principal,2160,JARDIM DAS CAMÉLIAS,273
16,1,SP,71072,SÃO PAULO,328,741,Principal,2364,PIRAJUSSARA,289
18,1,SP,71072,SÃO PAULO,349,426,Principal,1317,VILA NOVA GALVÃO,379
19,1,SP,71072,SÃO PAULO,260,288,Principal,1295,VILA INDEPENDENCIA,395
22,1,SP,71072,SÃO PAULO,422,337,Principal,1147,JARDIM VIEIRA DE CARVALHO,349
23,1,SP,71072,SÃO PAULO,420,242,Principal,1171,VILA ISOLINA MAZZEI,318
24,1,SP,71072,SÃO PAULO,403,409,Principal,1554,CONJUNTO RESIDENCIAL ELISIO TEIXEIRA LEITE,224


In [18]:
votos_path = '../dados/votos/bweb_1t_SP_051020261403.csv'
candidatos = [13, 22]
v_colunas = chave + [
    "NR_TURNO",
    "DS_CARGO_PERGUNTA",
    "DS_TIPO_VOTAVEL",
    "NR_VOTAVEL",
    "NM_VOTAVEL",
    "QT_VOTOS",
]


votos_por_candidato = Counter()
nomes_candidatos = set()

for bloco in pd.read_csv(votos_path, sep=';', encoding='latin-1', usecols=v_colunas, dtype={"CD_MUNICIPIO": str}, chunksize=100000):
    recorte = bloco.loc[(bloco['CD_MUNICIPIO'].eq(codigo_sao_paulo)) & 
                      (bloco['NR_TURNO'].eq(1) &
                       bloco['DS_CARGO_PERGUNTA'].eq("Presidente") &
                       bloco['DS_TIPO_VOTAVEL'].eq("Nominal") &
                       bloco['NR_VOTAVEL'].isin(candidatos))].copy()
    for _, row in recorte.iterrows():
        identificador = (row['SG_UF'], row['CD_MUNICIPIO'], row['NR_ZONA'], row['NR_SECAO'], row['NR_VOTAVEL'])
        if row['NR_VOTAVEL'] in candidatos:
            votos_por_candidato[identificador] += row['QT_VOTOS']
            nomes_candidatos.add(row['NM_VOTAVEL'])

votos_secoes_df = pd.DataFrame([(uf, municipio, zona, secao, candidato, votos) for (uf, municipio, zona, secao, candidato), votos in votos_por_candidato.items()], columns=chave + ["NR_VOTAVEL", "QT_VOTOS"])

votos_secoes_df.head(10)
print(f"Votos por candidato: {len(votos_secoes_df)} registros.")

Votos por candidato: 53366 registros.


In [37]:
votos_com_bairro = votos_secoes_df.merge(bairros[chave + ["NM_BAIRRO"]], on=chave, how='left', validate='many_to_one')

#print(f"Votos sem bairro: {votos_com_bairro['NM_BAIRRO'].isna().sum()}")
#print(f"Votos com bairro: {votos_com_bairro['NM_BAIRRO'].notna().sum()}")

votos_por_bairro = (votos_com_bairro.groupby(['NM_BAIRRO', 'NR_VOTAVEL'], as_index=False)['QT_VOTOS'].sum()
.sort_values(['NM_BAIRRO', 'NR_VOTAVEL'])
)

total_por_bairro = votos_por_bairro.groupby('NM_BAIRRO')['QT_VOTOS'].transform('sum')
votos_por_bairro['PERCENTUAL'] = (
    votos_por_bairro['QT_VOTOS']
    .div(total_por_bairro.where(total_por_bairro.ne(0)))
    .mul(100)
    .round(2)
)

votos_por_bairro.rename(columns={'NM_BAIRRO': 'Bairro', 'NR_VOTAVEL': 'Candidato', 'QT_VOTOS': 'Votos', 'PERCENTUAL': 'Percentual'}, inplace=True)

votos_por_bairro.head(10)

,Bairro,Candidato,Votos,Percentual
0,ACLIMAÇÃO,13,7592,54.33
1,ACLIMAÇÃO,22,6382,45.67
2,AGUA BRANCA,13,8495,53.26
3,AGUA BRANCA,22,7456,46.74
4,ALTO DA LAPA,13,4108,42.67
5,ALTO DA LAPA,22,5519,57.33
6,ALTO DA MOOCA,13,5190,37.11
7,ALTO DA MOOCA,22,8797,62.89
8,ALTO DA RIVIERA,13,2560,64.73
9,ALTO DA RIVIERA,22,1395,35.27


In [38]:
# Guardar os resultados em um CSV
output_path = '../dados/resultados/votos_por_bairro_SP.csv'
votos_por_bairro.to_csv(output_path, sep=';', index=False, encoding='utf-8')
print(f"Resultados salvos em: {output_path}")

Resultados salvos em: ../dados/resultados/votos_por_bairro_SP.csv


In [40]:
!pip install folium

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [folium]━━━━ 3/4 [folium]


In [42]:
# Visualização dos resultados
%run gerar_mapa_bairros.py


{
  "bairros_exibidos": 898,
  "bairros_sem_coordenadas": [
    "CONJUNTO HABITACIONAL TURISTICA"
  ],
  "locais_com_coordenadas_invalidas": 13,
  "metodo_posicionamento": "Mediana da latitude e longitude dos locais únicos de cada bairro.",
  "percentuais": "Preservados do CSV de resultados; não recalculados."
}
Mapa salvo em: /home/joaoricardo/ICD/1Turno_SP/dados/resultados/mapa_pontos_por_bairro_13_22.html
